In [ ]:
# hello
# MAZEN 5WL W OMAR BYTASSSSSSSSSSSS

# Introduction to Machine Learning — Assignment 2

## 0. Imports and reproducibility

We use NumPy for the implementations, pandas for readable result tables, and matplotlib for plots. Scikit-learn is used only for loading the required datasets, splitting the data, and the explicit reference comparisons requested in the assignment.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_breast_cancer, load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    confusion_matrix as sklearn_confusion_matrix,
    precision_recall_fscore_support
)

np.set_printoptions(precision=6, suppress=True)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("NumPy version:", np.__version__)
print("All randomness in this notebook uses random_state / default_rng seed 42.")

## Part A — Data and toolkit

### Task 1. Load the four datasets and apply the required 60 / 20 / 20 split

The assignment specifies:
1. Split 20% into the test set using `random_state=42`.
2. Split 25% of the remaining 80% into validation (which is 20% of the full dataset).
3. Use stratification for classification datasets, but not for Diabetes regression.

Dataset-specific preparation:
- Diabetes: `load_diabetes(scaled=False)`; keep the target in its original units.
- Breast Cancer: flip the labels so malignant = 1 and benign = 0.
- Digits: divide pixel values by 16; do not standardize them.
- Iris: use the original four features and three classes.

In [ ]:
# Load datasets exactly as described in the assignment.
diabetes = load_diabetes(scaled=False)
breast = load_breast_cancer()
digits = load_digits()
iris = load_iris()

datasets = {}

# Diabetes: regression, so stratify=None.
datasets["Diabetes"] = {
    "X": diabetes.data.astype(float),
    "y": diabetes.target.astype(float),
    "feature_names": list(diabetes.feature_names),
    "task": "regression"
}

# Breast Cancer: malignant must be the positive class (1).
datasets["Breast cancer"] = {
    "X": breast.data.astype(float),
    "y": (1 - breast.target).astype(int),
    "feature_names": list(breast.feature_names),
    "task": "classification"
}

# Digits: scale pixel intensities to [0, 1] by dividing by 16.
datasets["Digits"] = {
    "X": digits.data.astype(float) / 16.0,
    "y": digits.target.astype(int),
    "feature_names": [f"pixel_{i}" for i in range(digits.data.shape[1])],
    "task": "classification"
}

datasets["Iris"] = {
    "X": iris.data.astype(float),
    "y": iris.target.astype(int),
    "feature_names": list(iris.feature_names),
    "task": "classification"
}

# Apply the exact two-step split protocol.
splits = {}
for name, data in datasets.items():
    X, y = data["X"], data["y"]
    stratify_all = y if data["task"] == "classification" else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=stratify_all
    )

    stratify_train = y_train if data["task"] == "classification" else None
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.25, random_state=42,
        stratify=stratify_train
    )

    splits[name] = {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "feature_names": data["feature_names"],
        "task": data["task"]
    }

# Required split sizes.
expected_sizes = {
    "Diabetes": (264, 89, 89, 442),
    "Breast cancer": (341, 114, 114, 569),
    "Digits": (1077, 360, 360, 1797),
    "Iris": (90, 30, 30, 150)
}

rows = []
for name, sp in splits.items():
    sizes = tuple(len(sp[k]) for k in ["y_train", "y_val", "y_test"])
    total = sum(sizes)
    rows.append([name, *sizes, total, expected_sizes[name]])
    assert sizes + (total,) == expected_sizes[name], f"Unexpected split sizes for {name}"

split_table = pd.DataFrame(
    [[r[0], r[1], r[2], r[3], r[4]] for r in rows],
    columns=["Dataset", "Train", "Validation", "Test", "Total"]
)
display(split_table)
print("All split sizes match the assignment.")